# XTraffic ST-GNN — Fusion training (free T4 GPU)

Phase 6: retrains METR-LA with the **three real heterogeneous modalities**
(weather + events + transit) and lays the fusion model side by side with the
traffic-only baseline. Also **regenerates `metr_la_best.pt`** (the traffic-only
checkpoint) — handy since a local smoke run clobbered the old placeholder.

**Runtime → Change runtime type → T4 GPU** before running.

Checkpoints and the comparison table are written **directly to Google Drive**
(`MyDrive/xtraffic/`), so they survive the Colab tab closing or the runtime being
recycled — the reason the previous run left nothing behind.

In [ ]:
# CELL 1 — Mount Google Drive FIRST, before anything else, so every checkpoint
# lands on permanent storage. Colab local disk is wiped when the tab closes;
# Drive is not. This is the fix for 'training ran but no checkpoints survived'.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/xtraffic/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/xtraffic/results', exist_ok=True)
print('Drive mounted. Checkpoint dir ready.')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# CELL 2 — Clone the repo, then REDIRECT the checkpoint dir to Drive.
#
# train.py saves to xtraffic/models/gnn/checkpoints/<run_name>_best.pt (a path
# baked into the config, relative to the package). Rather than editing that code,
# we replace that local folder with a SYMLINK to Drive. Result: torch.save() inside
# training writes straight through to Drive as each best epoch lands — not just at
# the end — so a crashed tab still leaves you the best-so-far checkpoint.
REPO_URL = 'https://github.com/NarenS31/olympiflow'
CHECKPOINT_DIR = '/content/drive/MyDrive/xtraffic/checkpoints/'

!git clone $REPO_URL OlympiFlow
%cd OlympiFlow

# Point xtraffic/models/gnn/checkpoints at the Drive folder.
!rm -rf xtraffic/models/gnn/checkpoints
os.symlink(CHECKPOINT_DIR.rstrip('/'), 'xtraffic/models/gnn/checkpoints')
print('Checkpoints will be written to:', os.path.realpath('xtraffic/models/gnn/checkpoints'))

In [ ]:
# Pinned lightweight deps (training needs torch + numpy + pandas + pyyaml + matplotlib + requests).
!pip install -q pyyaml==6.0.1 requests==2.31.0 pandas matplotlib scipy

## 1. Build the traffic tensors + the three modality sidecars
Everything is downloaded/derived reproducibly — no manual files. The sidecars
are windowed and split *identically* to the traffic tensors, so sample i lines
up across all feeds.

In [ ]:
# Traffic tensors (Phase 1). Produces processed/metr_la/{train,val,test}.npz
!python -m xtraffic.data.pipelines.metr_la

In [ ]:
# Weather (Open-Meteo ERA5, keyless). NOTE: the archive API does not serve
# `visibility` -> that channel is all-NaN and gets zeroed (warned); temp+precip
# carry the signal. This is handled defensively, not a crash.
!python -m xtraffic.data.pipelines.weather --dataset metr_la

In [ ]:
# Events (committed curated 2012 venue schedule, proximity-decayed).
!python -m xtraffic.data.pipelines.events --dataset metr_la

In [ ]:
# Transit (LA Metro GTFS static -> nearby stop count per node).
!python -m xtraffic.data.pipelines.transit --dataset metr_la

## 2. Train the traffic-only baseline
Uses `train_metr_la.yaml` (`use_sidecars` off) -> `metr_la_best.pt`. This is the
apples-to-apples baseline AND restores the traffic-only checkpoint the rest of
the pipeline (Phases 3–5) loads. Saves directly to Drive via the symlink above.

In [ ]:
!python -m xtraffic.models.gnn.train --config configs/train_metr_la.yaml

## 3. Train the fusion model
Uses `train_metr_la_fusion.yaml` (`use_sidecars: true`, `run_name: metr_la_fusion`)
-> `metr_la_fusion_best.pt`. Same architecture and hyperparameters as the baseline;
the ONLY difference is the three modality feeds, so any gap is attributable to fusion.

In [ ]:
!python -m xtraffic.models.gnn.train --config configs/train_metr_la_fusion.yaml

## 4. Fusion vs traffic-only comparison table
Evaluates both checkpoints on the same test split at 15/30/60 min and reports the
learned per-modality gates (how much the model trusts each feed). Saved to
`evaluation/results/fusion/comparison.{csv,json}`.

In [ ]:
!python -m xtraffic.evaluation.fusion_comparison --dataset metr_la

## 5. (Optional) Baselines for the full prediction table
Historical-average + linear-regression reference numbers on the same splits.

In [ ]:
!python -m xtraffic.models.gnn.baselines --config configs/train_metr_la.yaml

## 6. Export the comparison table + results to Drive
The checkpoints are already on Drive (symlinked in Cell 2). Here we also copy the
comparison table and training logs onto Drive so the paper numbers are permanent too.

In [ ]:
# Copy the fusion-vs-traffic-only comparison + training summaries onto Drive.
import shutil, glob
results_path = '/content/drive/MyDrive/xtraffic/results/'
os.makedirs(results_path, exist_ok=True)

# The comparison table is the headline fusion result (save as fusion_vs_trafficonly.csv).
src_csv = 'xtraffic/evaluation/results/fusion/comparison.csv'
if os.path.exists(src_csv):
    shutil.copy(src_csv, os.path.join(results_path, 'fusion_vs_trafficonly.csv'))
    shutil.copy('xtraffic/evaluation/results/fusion/comparison.json',
                os.path.join(results_path, 'fusion_vs_trafficonly.json'))
    print('Saved fusion_vs_trafficonly.csv to Drive.')
else:
    print('WARNING: comparison.csv not found — did step 4 run?')

# Also copy per-run training logs (loss curves, epoch CSVs, summaries) for the paper.
for p in glob.glob('xtraffic/evaluation/results/train_metr_la*'):
    shutil.copy(p, os.path.join(results_path, os.path.basename(p)))
print('Results copied to:', results_path)

In [ ]:
# CELL (final) — VERIFICATION. This is how you KNOW the run actually finished and
# the checkpoints are safe on Drive. If this prints an empty list, training did NOT
# save — do not close the tab; scroll up and find the failing cell.
import os
ckpts = os.listdir('/content/drive/MyDrive/xtraffic/checkpoints/')
print('Checkpoints saved:', ckpts)
results = os.listdir('/content/drive/MyDrive/xtraffic/results/')
print('Results saved:', results)
assert any(c.endswith('_best.pt') for c in ckpts), 'NO _best.pt CHECKPOINT ON DRIVE — training did not complete!'
print('Training complete. Safe to close tab.')